In [ ]:
# !pip install msgpack
# !pip install autogluon

# 실습 2 - AutoGluon 모델 학습과 해석

**서울 공공자전거(따릉이) 데이터를 활용한 예측 모델 구축**

앞선 실습에서는 원본 데이터를 모델이 처리 가능한 수치로 변환하는 **전처리**를 다루었다. 본 실습에서는 해당 데이터로 **예측 모델을 학습**하고 결과를 해석한다.

본 실습에서 확인할 사항은 다음과 같다.

1. `fit()` 단일 명령으로 다수의 모델이 자동으로 학습된다.
2. AutoGluon은 문제 유형(회귀/분류)을 자동으로 판별한다.
3. 다수의 모델을 결합한 **앙상블**이 개별 모델보다 우수하다.
4. 어떤 변수가 예측에 중요한지 확인할 수 있다.

> 전처리(`fit_transform`)는 별도로 호출하지 않는다. `fit()` 내부에서 자동으로 수행되기 때문이다.


---
## 0. 환경 준비와 데이터 불러오기

필요한 라이브러리를 불러오고, 앞선 실습과 동일하게 따릉이 데이터를 불러온다.


In [ ]:
import sys
sys.path = [p for p in sys.path if ".local" not in p]
sys.path.insert(0, "/home/work/.local/lib/python3.10/site-packages")
import pandas as pd

df = pd.read_csv("SeoulBikeData.csv", encoding="latin-1")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

print("데이터 크기:", df.shape)
df.head(3)

---
## 1. 학습용 데이터와 평가용 데이터 분할

모델 구축 시 데이터를 두 부분으로 분할한다.

- **학습용(train)**: 모델 학습에 사용한다.
- **평가용(test)**: 학습에 사용하지 않고 별도로 보관하였다가, 완성된 모델의 성능 측정에 사용한다.

이와 같이 분할하는 이유는, 모델이 이미 학습한 데이터로 성능을 측정할 경우 실제보다 높게 측정되기 때문이다. 학습에 사용하지 않은 데이터에 대한 예측 성능을 확인해야 실제 성능(일반화 성능)을 파악할 수 있다.

아래 코드는 전체의 80%를 학습용, 20%를 평가용으로 무작위 분할한다.

여기서 분할하는 것은 최종 **평가용(test)** 이다. 이와 별개로, AutoGluon은 학습 과정에서 모델 비교를 위한 **검증용(validation)** 데이터를 학습용 내부에서 자동으로 분할한다. 즉, 연구자는 평가용만 분할하면 되며, 학습 과정의 검증 분할은 AutoGluon이 자동으로 처리한다.


In [ ]:
train_data = df.sample(frac=0.8, random_state=42)
test_data = df.drop(train_data.index)

print("학습용:", train_data.shape)
print("평가용:", test_data.shape)

In [ ]:
ratio = lambda d: d["Seasons"].value_counts(
            normalize=True).round(3) * 100

pd.DataFrame({
    "학습용 비율(%)": ratio(train_data),
    "평가용 비율(%)": ratio(test_data),
})

---
## 2. 모델 학습: `fit()`

모델을 학습한다. 방법은 단순하다.

- `TabularPredictor(label="Rented Bike Count")` : 예측 대상을 지정한다. `label`은 예측 목표 컬럼이며, 여기서는 대여량이다.
- `.fit(train_data)` : 학습용 데이터로 모델을 학습한다.

이 단일 명령 내에서 전처리, 다수 모델의 학습, 앙상블 구성이 모두 자동으로 수행된다.


In [ ]:
from autogluon.tabular import TabularPredictor
predictor = TabularPredictor(label="Rented Bike Count").fit(train_data)

#### 학습 로그에서 확인할 사항

1. **`AutoGluon infers your prediction problem is: 'regression'`**
   → 대여량이 수치형이고 값이 다양하므로, AutoGluon이 **회귀 문제**로 판별하였다. (예측 대상이 '고장/정상'과 같은 값이었다면 분류로 판별한다.)

2. **`Fitting model: ...`** 의 반복
   → LightGBM, RandomForest, CatBoost, 신경망(NeuralNet) 등 **다양한 모델이 순차적으로 학습**된다.

3. **`Fitting model: WeightedEnsemble_L2`**
   → 최종적으로 개별 모델들을 결합한 **앙상블 모델**이 생성된다.


---
## 3. 성능 확인: `leaderboard()`

학습된 모델들의 성능을 비교한다. `leaderboard(test_data)`는 평가용 데이터에 대한 각 모델의 성능을 측정하여 순위표로 출력한다.


In [ ]:
leaderboard = predictor.leaderboard(test_data)
leaderboard

- **`model`**: 모델 이름이다. `WeightedEnsemble`이 앞서 설명한 앙상블 모델이다.
- **`score_test`**: 평가용 데이터에 대한 성능이다. 회귀에서는 점수가 **0에 가까울수록(음수의 절댓값이 작을수록) 우수하다.** AutoGluon은 값이 클수록 우수하도록 부호를 통일하므로, 목록의 **최상단 행이 가장 우수한 모델**이다.
- 통상 최상단은 **`WeightedEnsemble`** 이다. 즉, 다수의 모델을 결합한 앙상블이 개별 모델보다 우수함을 확인할 수 있다.

> `score`가 음수인 이유: AutoGluon은 '클수록 우수함'으로 기준을 통일하기 위해, 오차 지표(작을수록 우수)에 음수 부호를 적용한다. 따라서 -138이 -145보다 우수한 성능이다.


---
## 4. 예측: `predict()`

완성된 모델로 평가용 데이터의 대여량을 예측하고 실제 값과 비교한다. `predict()`는 가장 우수한 모델(앙상블)을 자동으로 사용한다.


In [ ]:
predictions = predictor.predict(test_data)

# 실제 값과 예측 값을 나란히 비교
compare = pd.DataFrame({
    "실제 대여량": test_data["Rented Bike Count"].values,
    "예측 대여량": predictions.values.round(0)
})
compare.head(10)

실제 값과 예측 값의 근접 여부를 확인한다. 완전히 일치하지는 않으나, 대체로 근접한 값을 예측함을 확인할 수 있다.


---
## 5. 연구자가 조절하는 학습 설정

지금까지는 `fit()`을 **기본 설정**으로 실행하였다. AutoGluon은 개별 모델의 세부 하이퍼파라미터(학습률, 트리 깊이 등)를 연구자가 직접 조정할 필요가 없도록 설계되어 있다. 대신, 연구자는 다음 **세 가지 상위 설정**으로 학습의 방향을 결정한다.

| 설정 | 역할 | 값의 예 |
|---|---|---|
| `presets` | 성능과 학습 시간의 균형 | `best_quality`(고성능·느림) / `medium_quality`(빠름) |
| `time_limit` | 학습에 사용할 최대 시간(초) | `time_limit=600` (10분) |
| `eval_metric` | 성능 판단 기준 | 회귀: `rmse`, `mae` / 분류: `accuracy`, `f1` |

이 중 `eval_metric`은 **연구 목적에 따라 결정**된다. 무엇을 정확히 예측해야 하는가는 기계가 아니라 연구자가 판단하는 영역이다.


#### 예시: 평가지표(`eval_metric`) 변경 후 학습

기본값 대신 `mae`(평균 절대 오차)를 평가지표로 지정하여 재학습한다. `mae`는 예측이 실제 값에서 평균적으로 벗어난 정도를 절댓값으로 측정하는 지표이다. (학습 시간 단축을 위해 `time_limit`을 짧게 지정한다.)


In [ ]:
predictor_mae = TabularPredictor(
    label="Rented Bike Count",
    eval_metric="mae"              # 평가지표를 MAE로 지정 (기본값은 RMSE)
).fit(train_data)  # 학습 시간을 120초로 제한

predictor_mae.leaderboard(test_data)

`eval_metric` 열이 `mae`로 변경되었음을 확인할 수 있다. 평가 기준이 달라지면 모델의 순위나 최적 모델이 달라질 수 있다. 즉, 어떤 기준으로 우수한 모델을 선정하는가에 따라 결과가 달라진다.


#### 성능 우선 설정(`presets`)

더 높은 정확도가 필요한 경우 `presets="best_quality"`를 사용한다. 다만 다수의 모델을 더 많이, 더 깊게 학습하므로 시간이 오래 소요된다.


In [ ]:
predictor_best = TabularPredictor(
    label="Rented Bike Count"
).fit(train_data, 
      presets="best_quality", 
      fit_strategy="sequential",
      time_limit=600)

predictor_best.leaderboard(test_data)

#### 표 데이터용 파운데이션 모델

AutoGluon에는 **표(정형) 데이터 전용 파운데이션 모델**도 포함되어 있다. 대표적으로 **TabPFNv2**와 **Mitra**가 있으며, 대량의 표 데이터로 사전학습되어 있어 별도의 장시간 학습 없이도 예측이 가능하다. 특히 데이터 규모가 작을 때 우수한 성능을 보인다.

사용 시 유의사항은 다음과 같다.

- **GPU가 필요하다.** 신경망 기반이므로 CPU에서는 학습 속도가 매우 느리다.
- **별도 설치가 필요하다.** 기본 설치에는 포함되어 있지 않다.
- **회귀 문제**(따릉이 대여량 예측 등)에서는 `TabPFNv2`, `Mitra`를 사용할 수 있다. (`TabICL`은 분류 전용이다.)


In [ ]:
!{sys.executable} -m pip install tabpfn
!pip install omegaconf
!pip install autogluon.tabular[mitra]   # For Mitra
!pip install autogluon.tabular[tabicl]   # For TabICL
!pip install autogluon.tabular[tabpfn]   # For TabPFNv2

In [ ]:
predictor_fm = TabularPredictor(label="Rented Bike Count").fit(
    train_data,
    hyperparameters={
        "REALTABPFN-V2": {},   # 소규모 데이터에 강한 파운데이션 모델
        "MITRA": {},      # AutoGluon의 표 데이터 파운데이션 모델
    },
    num_gpus=1,
    num_cpus=16,
    verbosity=2
)

predictor_fm.leaderboard(test_data)

---
## 6. 변수 중요도 확인: `feature_importance()`

어떤 변수가 예측에 큰 영향을 미쳤는지 확인한다. 이는 모델의 예측 근거를 파악하는 과정으로, 연구에서 중요한 부분이다.


In [ ]:
importance = predictor.feature_importance(test_data)
importance

#### 결과 해석

- `importance` 값이 **클수록 해당 변수가 예측에 더 중요**함을 의미한다.
- 따릉이 데이터에서는 대체로 **`Hour`(시간)** 와 **`Temperature`(기온)** 의 중요도가 높게 나타난다. 이는 시간대와 기상 조건이 자전거 대여량을 결정하는 주요 요인임을 의미하며, 통념과도 부합한다.
- 이처럼 변수 중요도를 통해 연구자는 모델의 예측 근거를 해석하고 설명할 수 있다.


---
## 정리

| 단계 | 명령 | 내용 |
|---|---|---|
| 학습 | `TabularPredictor(label=...).fit()` | 다수 모델 학습 + 앙상블 자동 구성 |
| 성능 | `predictor.leaderboard(test)` | 모델별 성능 비교 (최상단이 최우수) |
| 예측 | `predictor.predict(test)` | 최우수 모델로 예측 |
| 설정 조절 | `presets`, `time_limit`, `eval_metric` | 성능·시간·평가기준을 연구자가 지정 |
| 해석 | `predictor.feature_importance(test)` | 변수별 중요도 확인 |

**요약:** `fit()` 단일 명령이 다수 모델의 학습과 앙상블을 자동으로 수행하며, 앙상블은 대체로 개별 모델보다 우수하다. 연구자는 결과(leaderboard, 변수 중요도)를 해석하고, 필요에 따라 설정(`presets`, `eval_metric`)을 조정한다. 특히 `eval_metric`은 연구 목적에 따라 결정되는 연구자의 판단 영역이다.
